# Notebook 1: Data and Representations

This notebook covers the first half of the dual-axis regime detection pipeline:
1. Data loading and exploratory analysis
2. Preprocessing: arcsinh transform and MSTL decomposition
3. Residual analysis
4. Sliding windows
5. Feature Engineering (15 features on Δr_t)
6. MOMENT embedding (1024-d on r_t)

The intermediate outputs are saved for use in **Notebook 2** (clustering and regime analysis).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings, gc, sys, os
from pathlib import Path
from scipy.stats import skew, kurtosis
from statsmodels.tsa.seasonal import MSTL

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

ROOT = Path('..').resolve()
OUT = ROOT / 'results_darcsinh' / 'split_W512_S6'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
W, S = 512, 6
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Output: {OUT}')

## 1. Load Data

The dataset contains 43,814 hourly Day-Ahead LMP observations from the Massachusetts Hub of ISO New England, spanning 2021–2025.

In [ ]:
df_raw = pd.read_parquet(ROOT / 'isone_dataset.parquet')
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])
print(f'Shape: {df_raw.shape}')
print(f'Period: {df_raw["datetime"].min().date()} to {df_raw["datetime"].max().date()}')
print(f'Missing: {df_raw["lmp"].isna().sum()}')
print(f'\nLMP statistics:')
df_raw['lmp'].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(df_raw['datetime'], df_raw['lmp'], lw=0.3, color='steelblue', alpha=0.8)
ax.set_ylabel('LMP ($/MWh)')
ax.set_title('ISO New England \u2014 Massachusetts Hub Day-Ahead LMP (2021\u20132025)')
ax.axhline(df_raw['lmp'].median(), color='orange', ls='--', lw=0.8, label=f'Median: ${df_raw["lmp"].median():.0f}')
ax.legend(frameon=False)
ax.set_xlim(df_raw['datetime'].min(), df_raw['datetime'].max())
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df_raw['lmp'], bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('LMP ($/MWh)')
axes[0].set_ylabel('Count')
axes[0].set_title('LMP Distribution')
axes[0].axvline(df_raw['lmp'].median(), color='orange', ls='--', lw=1, label='Median')
axes[0].axvline(df_raw['lmp'].mean(), color='red', ls='--', lw=1, label='Mean')
axes[0].legend(frameon=False, fontsize=8)

axes[1].hist(df_raw['lmp'], bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('LMP ($/MWh)')
axes[1].set_title('LMP Distribution (log scale)')
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

## 2. Preprocessing: arcsinh + MSTL

Two steps stabilize the series before regime detection:

1. **arcsinh transform**: `y_t = arcsinh(p_t)` — stabilizes variance, preserves full support (including potential negatives), acts as log for large values and is approximately linear near zero.

2. **MSTL decomposition**: removes three seasonal components (24h daily, 168h weekly, 8760h annual) and trend. The residual r_t retains the stochastic persistence that we want to analyze.

In [ ]:
lmp = df_raw['lmp'].values
dt = df_raw['datetime'].values
arcsinh_lmp = np.arcsinh(lmp)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(dt[:168*4], lmp[:168*4], lw=0.8, color='steelblue')
axes[0].set_title('Raw LMP (4 weeks)')
axes[0].set_ylabel('$/MWh')
axes[1].plot(dt[:168*4], arcsinh_lmp[:168*4], lw=0.8, color='darkgreen')
axes[1].set_title('arcsinh(LMP) (4 weeks)')
axes[1].set_ylabel('arcsinh($/MWh)')
plt.tight_layout()
plt.show()

In [ ]:
s = pd.Series(arcsinh_lmp, index=pd.DatetimeIndex(dt))
mstl_result = MSTL(s, periods=[24, 168, 8760]).fit()
resid = mstl_result.resid.values

print(f'Residual r_t: n={len(resid)}')
print(f'  Mean:     {resid.mean():.4f}')
print(f'  Std:      {resid.std():.3f}')
print(f'  Skew:     {skew(resid):.2f}')
print(f'  Kurtosis: {kurtosis(resid, fisher=False):.2f} (Pearson)')

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
components = [
    ('arcsinh(LMP)', s.values, 'steelblue'),
    ('Trend', mstl_result.trend.values, 'darkred'),
    ('Daily (24h)', mstl_result.seasonal.iloc[:, 0].values, 'teal'),
    ('Weekly (168h)', mstl_result.seasonal.iloc[:, 1].values, 'purple'),
    ('Annual (8760h)', mstl_result.seasonal.iloc[:, 2].values, 'goldenrod'),
    ('Residual r_t', resid, 'steelblue'),
]
for ax, (name, vals, color) in zip(axes, components):
    ax.plot(dt, vals, lw=0.3, color=color)
    ax.set_ylabel(name, fontsize=9)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('Date')
fig.suptitle('MSTL Decomposition of arcsinh(LMP)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Residual Analysis

The residual r_t has near-zero mean and dramatically lighter tails than raw LMP (kurtosis 4.50 vs 9.94). However, it retains very strong hourly persistence (ACF lag-1 = 0.977) — this is the *stochastic memory* of the market that we want to detect.

In [ ]:
def acf(x, lag):
    n = len(x); m = x.mean(); v = ((x - m)**2).sum()
    if v < 1e-15 or lag >= n: return 0.0
    return float(((x[:n-lag] - m) * (x[lag:] - m)).sum() / v)

lags = [1, 24, 168, 8760]
acf_raw = [acf(arcsinh_lmp, l) for l in lags]
acf_res = [acf(resid, l) for l in lags]

acf_df = pd.DataFrame({
    'Lag': ['1h', '24h', '168h', '8760h'],
    'arcsinh(LMP)': [f'{v:.3f}' for v in acf_raw],
    'Residual r_t': [f'{v:.3f}' for v in acf_res],
})
print('Autocorrelation before and after MSTL:')
print(acf_df.to_string(index=False))

In [ ]:
from statsmodels.tsa.stattools import acf as sm_acf

acf_vals = sm_acf(resid, nlags=200)
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(acf_vals)), acf_vals, width=1, color='steelblue', alpha=0.7)
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('ACF')
ax.set_title('Autocorrelation of residual r_t')
ax.axhline(0, color='black', lw=0.5)
for lag_mark in [24, 168]:
    ax.axvline(lag_mark, color='red', ls='--', lw=0.5, alpha=0.5)
    ax.text(lag_mark+2, 0.9, f'{lag_mark}h', fontsize=8, color='red')
plt.tight_layout()
plt.show()

## 4. Levels and Increments

The residual r_t can be viewed in two complementary ways:
- **Levels r_t** (persistent, ACF ≈ 0.977): encode how shocks propagate over time → input for MOMENT
- **Increments Δr_t = r_t − r_{t−1}** (stationary, ACF ≈ −0.5): encode the distribution of shocks → input for Feature Engineering

Both are deterministically derived from the same series — they are two views of the same phenomenon.

In [ ]:
dr = np.diff(resid)
r = resid[1:]
dt_a = dt[1:]
lmp_a = lmp[1:]

acf1_r = np.corrcoef(r[:-1], r[1:])[0, 1]
acf1_dr = np.corrcoef(dr[:-1], dr[1:])[0, 1]
print(f'r_t  (levels):     n={len(r)},  ACF lag-1 = {acf1_r:.3f}')
print(f'\u0394r_t (increments): n={len(dr)}, ACF lag-1 = {acf1_dr:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
sl = slice(5000, 5000+512)
axes[0].plot(r[sl], lw=0.8, color='steelblue')
axes[0].set_ylabel('r_t (levels)')
axes[0].set_title('Persistent: memory of shocks is visible')
axes[0].grid(alpha=0.2)
axes[1].plot(dr[sl], lw=0.8, color='darkgreen')
axes[1].set_ylabel('\u0394r_t (increments)')
axes[1].set_title('Stationary: no memory, shock sizes vary')
axes[1].set_xlabel('Hours within window')
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
df_pre = pd.DataFrame({'datetime': dt_a, 'lmp': lmp_a, 'r': r, 'dr': dr})
df_pre.to_parquet(OUT / 'preprocessed.parquet', index=False)
print(f'Saved preprocessed: {OUT / "preprocessed.parquet"}')

## 5. Sliding Windows

Each window contains W = 512 consecutive residual hours (≈ 21 days). Windows advance by S = 6 hours, producing N = 7,217 overlapping windows. The overlap (506 of 512 hours) provides fine temporal resolution for tracking regime transitions.

In [ ]:
def make_windows(vals, lmp_arr, dt_arr):
    starts = list(range(0, len(vals) - W + 1, S))
    wv = np.array([vals[s:s+W] for s in starts], dtype=np.float32)
    wl = np.array([lmp_arr[s:s+W] for s in starts], dtype=np.float32)
    ts = np.array([dt_arr[s+W-1] for s in starts])
    return wv, wl, ts

wr_fe, wl, ts = make_windows(dr, lmp_a, dt_a)  # FE gets \u0394r_t
wr_mom, _, _ = make_windows(r, lmp_a, dt_a)     # MOMENT gets r_t
N = len(wr_fe)
print(f'Windows: N = {N:,}')
print(f'Window size: W = {W} hours (\u2248 {W/24:.0f} days)')
print(f'Stride: S = {S} hours')
print(f'Overlap: {W-S} hours ({100*(W-S)/W:.1f}%)')
print(f'First window ends: {ts[0]}')
print(f'Last window ends:  {ts[-1]}')

## 6. Feature Engineering (15 features on Δr_t)

For each window, we compute 15 hand-crafted features:
- 11 distributional statistics on the stationary increments Δr_t
- 1 intraday volatility measure on Δr_t
- 3 raw LMP statistics on the original prices p_t (reintroducing the price level removed by preprocessing)

These features are order-invariant: they describe *what* happens in the window (shock sizes, price level) but not *how* the shocks evolve over time.

In [ ]:
FE_NAMES = ['mean', 'std', 'skew', 'kurt', 'min', 'max', 'range',
            'median', 'p5', 'p95', 'iqr', 'vol_24h',
            'lmp_mean', 'lmp_p95', 'lmp_std']

def compute_fe(wr, wl):
    n = len(wr)
    fe = np.empty((n, 15), dtype=np.float32)
    for i in range(n):
        x = wr[i].astype(np.float64)
        p = wl[i].astype(np.float64)
        fe[i, 0] = x.mean()
        fe[i, 1] = x.std()
        fe[i, 2] = float(skew(x))
        fe[i, 3] = float(kurtosis(x, fisher=False))
        fe[i, 4] = x.min()
        fe[i, 5] = x.max()
        fe[i, 6] = x.max() - x.min()
        fe[i, 7] = float(np.median(x))
        fe[i, 8] = float(np.percentile(x, 5))
        fe[i, 9] = float(np.percentile(x, 95))
        fe[i, 10] = float(np.percentile(x, 75) - np.percentile(x, 25))
        nf = (len(x) // 24) * 24
        fe[i, 11] = float(np.abs(np.diff(x[:nf].reshape(-1, 24), axis=1)).mean()) if nf >= 24 else float(np.abs(np.diff(x)).mean())
        fe[i, 12] = p.mean()
        fe[i, 13] = float(np.percentile(p, 95))
        fe[i, 14] = p.std()
    return fe

fe = compute_fe(wr_fe, wl)
print(f'FE matrix: {fe.shape}')

df_fe = pd.DataFrame(fe, columns=FE_NAMES)
df_fe['datetime'] = ts
print(f'\nFeature summary:')
df_fe[FE_NAMES].describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_fe[FE_NAMES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
df_fe.to_parquet(OUT / 'fe_features.parquet', index=False)
print(f'Saved: {OUT / "fe_features.parquet"}')

## 7. MOMENT Embedding (1024-d on r_t)

MOMENT-1-large is a time series foundation model (340M parameters) pre-trained on 385,000+ diverse time series. Applied **zero-shot** (no fine-tuning), it receives the persistent residual r_t and produces a 1024-dimensional embedding per window.

The key insight: MOMENT receives only the deseasoned residual r_t, which oscillates around zero regardless of price level. Any structure it finds comes from the *temporal dynamics* of the series, not from price level.

**Note**: This cell requires a GPU for reasonable performance (~5 min on a single GPU, much slower on CPU).

In [ ]:
from momentfm import MOMENTPipeline

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Loading MOMENT-1-large on {DEVICE}...')
model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large', model_kwargs={'task_name': 'embedding'})
model.init()
model = model.to(DEVICE)

BS = 64
with torch.no_grad():
    d_model = model(x_enc=torch.zeros(1, 1, W, device=DEVICE)).embeddings.shape[-1]
print(f'Embedding dimension: {d_model}')

mom = np.empty((N, d_model), dtype=np.float32)
for s in range(0, N, BS):
    e = min(s + BS, N)
    x = torch.tensor(wr_mom[s:e], dtype=torch.float32, device=DEVICE).unsqueeze(1)
    with torch.no_grad():
        mom[s:e] = model(x_enc=x).embeddings.float().cpu().numpy()
    if (s // BS) % 20 == 0:
        print(f'  {e:,}/{N:,}')

del model, x
gc.collect()
torch.cuda.empty_cache()
print(f'\nMOMENT embeddings: {mom.shape}')

In [ ]:
norms = np.linalg.norm(mom, axis=1)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(pd.to_datetime(ts), norms, lw=0.3, color='steelblue')
ax.set_ylabel('L2 norm')
ax.set_title('MOMENT Embedding Norms Over Time')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()
print(f'Norm stats: mean={norms.mean():.1f}, std={norms.std():.1f}, min={norms.min():.1f}, max={norms.max():.1f}')

In [ ]:
df_mom = pd.DataFrame(mom, columns=[f'mom_{i}' for i in range(mom.shape[1])])
df_mom['datetime'] = ts
df_mom.to_parquet(OUT / 'moment_embeddings.parquet', index=False)
print(f'Saved: {OUT / "moment_embeddings.parquet"}')

## Summary

Notebook 1 has produced:
- `preprocessed.parquet`: 43,813 hourly observations with r_t and Δr_t
- `fe_features.parquet`: 7,217 windows × 15 features
- `moment_embeddings.parquet`: 7,217 windows × 1,024 embedding dimensions

Continue to **Notebook 2** for Diffusion Maps, ToMATo clustering, Tukey HSD merge, and the two-axis regime analysis.

In [ ]:
print('Files saved:')
for f in sorted(OUT.glob('*.parquet')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}: {size_mb:.1f} MB')